In [2]:
import pandas as pd

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
movies = pd.read_csv("../data/featured_movie.csv")

In [4]:
movies.head(3)

,id,original_title,tags
0,19995,Avatar,action adventure fantasy science fiction cultu...
1,285,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drug abuse exot...
2,206647,Spectre,action adventure crime spy based on novel secr...


In [5]:
movies['tags'].isnull().sum()

np.int64(0)

In [6]:
movies["tags"].head()

0    action adventure fantasy science fiction cultu...
1    adventure fantasy action ocean drug abuse exot...
2    action adventure crime spy based on novel secr...
3    action crime drama thriller dc comics crime fi...
4    action adventure science fiction based on nove...
Name: tags, dtype: str

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

In [10]:
tfidf_matrix = tfidf.fit_transform(movies["tags"])

In [11]:
tfidf_matrix.shape

(4800, 5000)

In [14]:
tfidf.get_feature_names_out()[62]

'action'

In [13]:
print(tfidf_matrix[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 46 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 62)	0.054276554958356836
  (0, 91)	0.06268234558713302
  (0, 1625)	0.07677638967155379
  (0, 3942)	0.07208962367314474
  (0, 1671)	0.07237940635785312
  (0, 1076)	0.1321356768426647
  (0, 845)	0.1394775690340491
  (0, 1809)	0.10257649979337909
  (0, 4192)	0.4289797496070234
  (0, 4826)	0.1543681734857762
  (0, 902)	0.14913157246638334
  (0, 4161)	0.11778427129279599
  (0, 4608)	0.10487791434875789
  (0, 1810)	0.14476969939303536
  (0, 3827)	0.06020619517632152
  (0, 148)	0.31567943187501935
  (0, 4624)	0.15452842974886977
  (0, 3401)	0.11519692875639384
  (0, 2805)	0.2497304829046102
  (0, 4167)	0.11465971739911769
  (0, 398)	0.09767045949293082
  (0, 2725)	0.07050077894568209
  (0, 95)	0.11659800814722439
  (0, 225)	0.12404978229309277
  (0, 3463)	0.10453459315291877
  (0, 3685)	0.1596228928186131
  (0, 2946)	0.12365272134088388
  (0, 4184)	0.1304845650922829
  (

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [18]:
similarity_matrix.shape

(4800, 4800)

In [19]:
similarity_matrix[:5, :5]

array([[1.        , 0.02149984, 0.02250744, 0.01443606, 0.21627591],
       [0.02149984, 1.        , 0.01020514, 0.01657073, 0.04659466],
       [0.02250744, 0.01020514, 1.        , 0.03053767, 0.02330282],
       [0.01443606, 0.01657073, 0.03053767, 1.        , 0.00915248],
       [0.21627591, 0.04659466, 0.02330282, 0.00915248, 1.        ]])

In [21]:
movie_index = movies[
    movies["original_title"] == "Interstellar"
].index[0]

In [22]:
similarity_scores = similarity_matrix[movie_index]

In [23]:
similarity_scores

array([0.20593911, 0.00444964, 0.00449683, ..., 0.01976688, 0.        ,
       0.        ], shape=(4800,))

In [27]:
sorted_movies = sorted(
    list(enumerate(similarity_scores)),
    key=lambda x: x[1],
    reverse=True
)

In [28]:
for index, score in sorted_movies[:10]:
    print(
        movies.iloc[index]["original_title"],
        score
    )

Interstellar 1.0
Moonraker 0.30455943257816176
Silent Running 0.29453014323675175
2001: A Space Odyssey 0.27985378632313446
Cargo 0.2735128518927005
Lost in Space 0.27191519478189713
Armageddon 0.2660905751125724
Gravity 0.26016696986302074
キャプテンハーロック 0.25460193442359585
Mission to Mars 0.2543234876669552


In [32]:
movies["normalized_title"] = (
    movies["original_title"]
    .str.strip()
    .str.lower()
)

In [33]:
def recommend_movies(movie_title, n=5):

    # Validate movie title
    if not isinstance(movie_title, str):
        return []

    # Validate number of recommendations
    if not isinstance(n, int) or n <= 0:
        return []

    # Normalize user input
    movie_title = movie_title.strip().lower()

    # Find matching movie
    matches = movies[
        movies["normalized_title"] == movie_title
    ]

    # Movie not found
    if matches.empty:
        return []

    # Select first matching movie
    movie_index = matches.index[0]

    # Get similarity scores
    similarity_scores = similarity_matrix[movie_index]

    # Sort by similarity
    sorted_movies = sorted(
        list(enumerate(similarity_scores)),
        key=lambda x: x[1],
        reverse=True
    )

    # Collect recommendations
    recommendations = []

    for index, score in sorted_movies[1:n + 1]:

        recommendations.append(
            movies.iloc[index]["title"]
        )

    return recommendations